# 03 — Train one model and inspect predictions
Run All independently with shared VDD/configuration. Default: 12 epochs, 16 frames at 16 FPS, 32×32 input, batch 8. Test stays locked. Exploratory diagnostics, not paper results.

In [ ]:
from pathlib import Path
import subprocess
import sys
import importlib

REPO_ROOT = Path('/content/phd_research')
if not REPO_ROOT.is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/sanelehlabisa/phd_research.git', str(REPO_ROOT)], check=True)
else:
    update = subprocess.run(['git', 'pull', '--ff-only', 'origin', 'master'], cwd=REPO_ROOT, capture_output=True, text=True)
    print(update.stdout, flush=True)
    if update.returncode:
        raise RuntimeError('Cannot safely update the Colab checkout. Save/commit conflicting local edits, resolve the Git error, then rerun setup. No local edits were discarded.\n' + update.stderr)
IMPLEMENTATION_ROOT = REPO_ROOT / 'papers/001-journal-abnormal-activity-recognition/implementation'
required_helpers = ('notebook_data.py', 'notebook_display.py', 'notebook_workflows.py')
missing_helpers = [name for name in required_helpers if not (IMPLEMENTATION_ROOT / 'src' / name).is_file()]
revision = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_ROOT, capture_output=True, text=True, check=True).stdout.strip()
print(f'Code checkout: {REPO_ROOT} @ {revision}', flush=True)
if missing_helpers:
    raise RuntimeError(f'Colab checkout is missing repository files: {missing_helpers}. These are NOT pip packages. Commit and push the complete notebook/source update to master, then rerun this setup cell. A kernel restart alone cannot download unpublished files.')
bootstrap = subprocess.run([sys.executable, str(IMPLEMENTATION_ROOT / 'src/colab_bootstrap.py'), str(IMPLEMENTATION_ROOT / 'requirements.txt')])
if bootstrap.returncode == 75:
    raise SystemExit('Restart the Colab runtime, reconnect to the A100, and rerun from the top.')
bootstrap.check_returncode()
sys.path[:] = [str(IMPLEMENTATION_ROOT)] + [entry for entry in sys.path if entry != str(IMPLEMENTATION_ROOT)]
for name in [name for name in sys.modules if name == 'src' or name.startswith('src.')]:
    del sys.modules[name]
importlib.invalidate_caches()
from src.notebook_utils import validate_runtime
print(validate_runtime(require_cuda=True))
from src.notebook_data import prepare_data
prepared = prepare_data(IMPLEMENTATION_ROOT)


## Train and validate
Batch progress, live epoch loss/accuracy curves and saved run artifacts. Retain the lowest-validation-loss checkpoint.

In [ ]:
from src.notebook_workflows import train_single
training_result = None  # Do not reuse an old result if training fails.
training_result = train_single(prepared)


## Selected-checkpoint predictions
Five validation clips (or the entire partition if smaller), with playable video, labels, correctness and probabilities.

In [ ]:
from src.notebook_workflows import show_validation_predictions
validation_examples = show_validation_predictions(prepared, training_result)
